# Paso 4-5 — Limpieza y union de los datos de establecimientos educativos

Aplica las reglas documentadas en `reports/plan_limpieza.md` (que a su vez se
basa en `reports/diagnostico_datos_crudos.md`, generado por `04_diagnostico.py`)
sobre la union de los 23 CSV crudos de `data/crudo/`.

Regla general: **no se modifica la ortografia, tildes, ni mayusculas/minusculas
originales de los campos de texto libre** (ESTABLECIMIENTO, DIRECCION,
SUPERVISOR, DIRECTOR). Solo se corrigen problemas de formato (espacios,
placeholders de "sin dato") y se documenta cada decision con su justificacion
y sus riesgos.

In [2]:
import glob
import re
from pathlib import Path

import pandas as pd

CRUDO_DIR = Path('data/crudo')
LIMPIO_DIR = Path('data/limpio')

archivos = sorted(glob.glob(str(CRUDO_DIR / '*.csv')))
df = pd.concat([pd.read_csv(f, dtype=str, encoding='utf-8-sig') for f in archivos], ignore_index=True)
print(f'Archivos cargados: {len(archivos)}')
print(f'Shape inicial (crudo, sin limpiar): {df.shape}')
df.head(3)

Archivos cargados: 23
Shape inicial (crudo, sin limpiar): (91355, 17)


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,00-01-0001-42,01-01-0030,CIUDAD CAPITAL,ZONA 1,EODP NO.20 'ANTONIO JOSE DE IRISARRI',19 CALLE 12-71,22324443,NORMA ARACELY PALOMO FRANCO DE DÍAZ,NaN,PARVULOS,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
1,00-01-0002-42,01-01-0030,CIUDAD CAPITAL,ZONA 1,EODP NO.21 GABRIELA MISTRAL,19 CALLE 12-71,22324443,NORMA ARACELY PALOMO FRANCO DE DÍAZ,LUISA IRENE SERRANO ESCOBAR,PARVULOS,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
2,00-01-0003-42,01-01-0030,CIUDAD CAPITAL,ZONA 1,EODP NO. 36 LA PRESIDENTA,2A AVE. Y 22 CALLE TERCER NIVEL MERCADO LA PRE...,55161115,NORMA ARACELY PALOMO FRANCO DE DÍAZ,MANDY YENEDITH HERNÁNDEZ BOBADILLA,PARVULOS,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE


## CODIGO

**Problema:** ninguno (0% faltante, 100% cumple `NN-NN-NNNN-NN`, es llave unica).
**Regla:** se deja tal cual como texto y se usa como llave primaria.
**Riesgo:** ninguno; solo se revalida la unicidad.

In [ ]:
codigo_pat = re.compile(r'^\d{2}-\d{2}-\d{4}-\d{2}$')
assert df['CODIGO'].str.match(codigo_pat).all(), 'Hay CODIGO fuera de formato'
assert df['CODIGO'].is_unique, 'CODIGO no es unico'
print('CODIGO: formato 100% valido y unico. Sin cambios.')

## DISTRITO — formato incompleto a NaN

**Problema:** conviven dos formatos legitimos (`NN-NN-NNNN` y `NN-NNN`) y 79
filas con un formato incompleto (`NN-`, sin nada despues) que no aporta
informacion utilizable.

**Regla:** no se fuerza un unico formato (no hay certeza de que ambos
esquemas sean intercambiables); solo se convierten a NaN los 79 casos
realmente vacios de contenido.

**Riesgo:** dejar dos formatos sin unificar puede complicar un join futuro
contra otra fuente que use un solo esquema de distrito; queda documentado.
incompleto = df['DISTRITO'].str.match(r'^\d{2}-$', na=False)
print(f'DISTRITO incompleto (NN- sin nada mas): {incompleto.sum()} filas -> se convierten a NaN')
df.loc[incompleto, 'DISTRITO'] = pd.NA

In [3]:
incompleto = df['DISTRITO'].str.match(r'^\d{2}-$', na=False)
print(f'DISTRITO incompleto (NN- sin nada mas): {incompleto.sum()} filas -> se convierten a NaN')
df.loc[incompleto, 'DISTRITO'] = pd.NA

DISTRITO incompleto (NN- sin nada mas): 74 filas -> se convierten a NaN


## DEPARTAMENTO y MUNICIPIO — unificar Ciudad Capital con Guatemala

**Problema:** el sitio trata `CIUDAD CAPITAL` como un departamento aparte de
`GUATEMALA` (23 valores en vez de los 22 departamentos oficiales); bajo
Ciudad Capital, `MUNICIPIO` trae `ZONA 1`..`ZONA 25` en vez de un municipio
real. No hay solapamiento de CODIGO entre ambos grupos (verificado en el
diagnostico).

**Regla:** unificar `DEPARTAMENTO = CIUDAD CAPITAL` dentro de
`DEPARTAMENTO = GUATEMALA`, fijar `MUNICIPIO = GUATEMALA` para esas filas, y
conservar la zona en una columna nueva `ZONA` (en vez de depender de
DIRECCION, que solo la menciona en el 16% de los casos).

**Por que funcionara:** el dataset queda alineado con los 22 departamentos
oficiales de Guatemala sin perder la informacion de zona (extraccion
verificada al 100%, 0 zonas sin capturar).

**Riesgo:** `ZONA` es una columna nueva que no viene del formulario
original; se documenta explicitamente en el libro de codigos para que no se
confunda con una variable oficial de MINEDUC.

In [4]:
es_capital = df['DEPARTAMENTO'] == 'CIUDAD CAPITAL'
print(f'Filas bajo CIUDAD CAPITAL a reclasificar: {es_capital.sum()}')

df['ZONA'] = pd.NA
zona_extraida = df.loc[es_capital, 'MUNICIPIO'].str.extract(r'ZONA\s*(\d+)')[0]
assert zona_extraida.notna().all(), 'Hay filas de Ciudad Capital sin zona capturada'
df.loc[es_capital, 'ZONA'] = zona_extraida

df.loc[es_capital, 'MUNICIPIO'] = 'GUATEMALA'
df.loc[es_capital, 'DEPARTAMENTO'] = 'GUATEMALA'

# Mover ZONA junto a MUNICIPIO para que el orden de columnas sea legible.
cols = df.columns.tolist()
cols.insert(cols.index('MUNICIPIO') + 1, cols.pop(cols.index('ZONA')))
df = df[cols]

print(f'DEPARTAMENTO ahora tiene {df["DEPARTAMENTO"].nunique()} valores (esperado: 22)')
df.loc[df['ZONA'].notna(), ['DEPARTAMENTO', 'MUNICIPIO', 'ZONA']].head(3)

Filas bajo CIUDAD CAPITAL a reclasificar: 4413
DEPARTAMENTO ahora tiene 22 valores (esperado: 22)


,DEPARTAMENTO,MUNICIPIO,ZONA
0,GUATEMALA,GUATEMALA,1
1,GUATEMALA,GUATEMALA,1
2,GUATEMALA,GUATEMALA,1


## ESTABLECIMIENTO, DIRECCION, SUPERVISOR — solo espacios dobles

**Problema:** miles de filas con espacios internos multiples.

**Regla:** colapsar espacios multiples a uno solo. **No se toca ortografia,
tildes ni mayusculas/minusculas** — instruccion explicita del proyecto,
porque estos textos se usan despues en informes.

**Riesgo:** minimo (un nombre con doble espacio intencional es
extremadamente improbable en este dominio).

In [5]:
def colapsar_espacios(serie):
    return serie.str.replace(r'\s+', ' ', regex=True).str.strip()

for col in ['ESTABLECIMIENTO', 'DIRECCION', 'SUPERVISOR']:
    antes = df[col].fillna('').str.contains(r'  +').sum()
    df[col] = colapsar_espacios(df[col])
    print(f'{col}: {antes} filas con espacios dobles corregidas')

ESTABLECIMIENTO: 4328 filas con espacios dobles corregidas
DIRECCION: 2009 filas con espacios dobles corregidas
SUPERVISOR: 317 filas con espacios dobles corregidas


## TELEFONO — formato uniforme de 8 digitos

**Problema:** mezcla de telefonos validos (8 digitos), valores `S/D`,
multiples numeros separados por `/` o `-`, y numeros con longitud distinta
de 8.

**Regla:** `S/D` (y equivalentes) -> NaN. Si hay varios numeros juntos, se
conserva el primer numero de 8 digitos encontrado. Si no hay ningun tramo de
8 digitos exactos, se deja NaN (no se completa ni se trunca).

**Por que funcionara:** deja la columna en un formato uniforme y usable, sin
adivinar digitos que no estan.

**Riesgo:** se pierde el segundo numero en registros con mas de un
telefono, y se descartan numeros con longitud incorrecta que quiza tenian un
solo digito mal capturado (decision conservadora: preferible no adivinar).

In [6]:
SD_EQUIVALENTES = {'S/D', 'SIN DATO', 'N/A'}


def limpiar_telefono(valor):
    if pd.isna(valor) or valor in SD_EQUIVALENTES:
        return pd.NA
    match = re.search(r'\d{8}', valor)
    return match.group(0) if match else pd.NA


antes_validos = df['TELEFONO'].notna().sum()
df['TELEFONO'] = df['TELEFONO'].map(limpiar_telefono)
despues_validos = df['TELEFONO'].notna().sum()
print(f'TELEFONO no nulos antes: {antes_validos}, despues de limpiar: {despues_validos}')
print('(la baja es esperada: S/D y numeros sin un tramo de 8 digitos pasan a NaN)')

TELEFONO no nulos antes: 64361, despues de limpiar: 63895
(la baja es esperada: S/D y numeros sin un tramo de 8 digitos pasan a NaN)
